# Skin Lesion — Data Preprocessing and MLP Training

**Student:** !TODO

**Group:** Group 1 — Informatyka, Uczenie maszynowe

**Course:** Machine Learning

**Date:** 2026-06-05

This notebook prepares the HAM10000/ISIC training and test sets, preprocesses images for an MLP, trains a model, and evaluates with confusion matrix and weighted F1-score. The final cell is prepared for evaluation on the hidden test set.

## 1. Imports & Setup

In [ ]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Flatten, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, f1_score, classification_report

USE_GPU = len(tf.config.list_physical_devices('GPU')) > 0
DEVICE_NAME = '/GPU:0' if USE_GPU else '/CPU:0'
print(f"Using {'GPU' if USE_GPU else 'CPU'}: {DEVICE_NAME}")
print(f"TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRAIN_META    = 'data/HAM10000_metadata'
TEST_META     = 'data/ISIC2018_Task3_Test_GroundTruth.csv'
TRAIN_FOLDERS = ['data/HAM10000_images_part_1', 'data/HAM10000_images_part_2']
TEST_FOLDER   = 'data/ISIC2018_Task3_Test_Images'

# ── Image parameters ───────────────────────────────────────────────────────
IMG_WIDTH  = 128
IMG_HEIGHT = 128

# ── PCA ────────────────────────────────────────────────────────────────────
PCA_COMPONENTS = 256  # Retains ~95% variance, drastically reduces overfitting

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS      = 100
BATCH_SIZE  = 64 if USE_GPU else 32
RANDOM_SEED = 42
VAL_SPLIT   = 0.2

# ── Artifact paths ─────────────────────────────────────────────────────────
MODEL_PATH  = 'best_mlp_model.keras'
PCA_PATH    = 'pca.pkl'

# Class names
CLASS_NAMES = ['Melanocytic nevi (nv)', 'Melanoma (mel)', 'Others']
NUM_CLASSES = 3

## 3. Data Loading

In [ ]:
train_df = pd.read_csv(TRAIN_META)
test_df  = pd.read_csv(TEST_META)

print("Train columns:", train_df.columns.tolist())
print("Test  columns:", test_df.columns.tolist())
train_df.head()

In [ ]:
def build_path_map(folders):
    paths = {}
    for folder in folders:
        for f in glob.glob(os.path.join(folder, '*.jpg')):
            paths[os.path.splitext(os.path.basename(f))[0]] = f
    return paths

train_path_map = build_path_map(TRAIN_FOLDERS)
test_path_map  = build_path_map([TEST_FOLDER])

train_df['path'] = train_df['image_id'].map(train_path_map.get)
test_df['path']  = test_df['image_id'].map(test_path_map.get)

train_df = train_df.dropna(subset=['path'])
test_df  = test_df.dropna(subset=['path'])

print(f"Train images found: {len(train_df)}")
print(f"Test  images found: {len(test_df)}")

def map_labels(dx):
    if dx == 'nv':  return 0  # Melanocytic nevi
    elif dx == 'mel': return 1  # Melanoma
    else: return 2  # Others

train_df['label'] = train_df['dx'].apply(map_labels)
test_df['label']  = test_df['dx'].apply(map_labels)

print("\nTrain label distribution:")
print(train_df['label'].value_counts().sort_index())
print("\nTest label distribution:")
print(test_df['label'].value_counts().sort_index())

## 4. Data Preprocessing

- **PCA** to 256 components — reduces 49 152 raw features, limits overfitting, speeds training
- **Data augmentation** applied to training set only to improve generalisation

In [ ]:
def load_images_rgb(df, augment=False, augment_minority_factor=3):
    """
    Load images as RGB arrays normalised to [0, 1].
    When augment=True, minority classes (mel, others) are augmented
    augment_minority_factor times using random flips, rotations, etc.
    Returns flat arrays (N, H*W*3) and label vector.
    """
    datagen = ImageDataGenerator(
        horizontal_flip=True,
        vertical_flip=True,
        rotation_range=20,
        zoom_range=0.1,
        width_shift_range=0.1,
        height_shift_range=0.1,
        brightness_range=[0.85, 1.15],
    )

    images, labels = [], []

    for _, row in df.iterrows():
        try:
            img = Image.open(row['path']).convert('RGB')
            img = img.resize((IMG_WIDTH, IMG_HEIGHT))
            arr = np.array(img, dtype=np.float32) / 255.0  # (H, W, 3)
            images.append(arr)
            labels.append(row['label'])

            # Augment minority classes (mel=1, others=2)
            if augment and row['label'] in [1, 2]:
                arr_4d = arr[np.newaxis]  # (1, H, W, 3)
                aug_iter = datagen.flow(arr_4d, batch_size=1)
                for _ in range(augment_minority_factor):
                    aug_img = next(aug_iter)[0]  # (H, W, 3)
                    aug_img = np.clip(aug_img, 0.0, 1.0)
                    images.append(aug_img)
                    labels.append(row['label'])

        except Exception as e:
            print(f"Error loading {row['path']}: {e}")

    images = np.array(images, dtype=np.float32)  # (N, H, W, 3)
    labels = np.array(labels, dtype=np.int32)

    images_flat = images.reshape(len(images), -1)
    return images_flat, labels


print("Loading & augmenting training images (this may take a few minutes)...")
X_train_raw, y_train = load_images_rgb(train_df, augment=False)

print("Loading test images...")
X_test_raw, y_test = load_images_rgb(test_df, augment=False)

print(f"\nX_train_raw shape : {X_train_raw.shape}")
print(f"X_test_raw  shape : {X_test_raw.shape}")
print(f"Train label distribution after augmentation: {np.bincount(y_train)}")

In [ ]:
print(f"Fitting PCA ({PCA_COMPONENTS} components) on training data...")
pca = PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_raw)
X_test_pca  = pca.transform(X_test_raw)

explained = np.sum(pca.explained_variance_ratio_) * 100
print(f"PCA explains {explained:.1f}% of variance with {PCA_COMPONENTS} components")
print(f"X_train_pca shape: {X_train_pca.shape}")
print(f"X_test_pca  shape: {X_test_pca.shape}")

with open(PCA_PATH, 'wb') as f:
    pickle.dump(pca, f)
print(f"PCA saved to '{PCA_PATH}'")

## 5. Train / Validation Split

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_pca, y_train,
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED,
    stratify=y_train
)

y_tr_cat  = to_categorical(y_tr,  num_classes=NUM_CLASSES)
y_val_cat = to_categorical(y_val, num_classes=NUM_CLASSES)

cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight_dict = {i: w for i, w in enumerate(cw)}

print(f"Train split : {X_tr.shape}")
print(f"Val   split : {X_val.shape}")
print(f"Class weights: {class_weight_dict}")

## 6. Model Design

Architecture is a 4-layer MLP tuned for PCA-reduced input (256 features). Deeper networks with wide early layers overfit on flat image features; this slimmer design generalises better.

In [ ]:
def build_mlp(input_dim, num_classes):
    model = Sequential([
        Input(shape=(input_dim,)),

        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),

        Dense(64, activation='relu'),
        BatchNormalization(),

        Dense(32, activation='relu'),
        BatchNormalization(),

        Dense(num_classes, activation='softmax')
    ])
    return model

with tf.device(DEVICE_NAME):
    model = build_mlp(PCA_COMPONENTS, NUM_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Model Training

**Hyperparameters:**
- Optimizer: Adam, initial LR = 1e-3
- LR schedule: ReduceLROnPlateau (factor=0.5, patience=5, min_lr=1e-6)
- Early stopping: patience=15 (gives LR scheduler time to recover)
- Batch size: 64 (GPU) / 32 (CPU)
- Class weights: balanced (sklearn)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH, monitor='val_loss',
        save_best_only=True, verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
]

print("Starting training...")
with tf.device(DEVICE_NAME):
    history = model.fit(
        X_tr, y_tr_cat,
        validation_data=(X_val, y_val_cat),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

print("Training complete.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train loss')
axes[0].plot(history.history['val_loss'], label='Val loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train acc')
axes[1].plot(history.history['val_accuracy'], label='Val acc')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Model Evaluation on Test Set

In [ ]:
best_model = load_model(MODEL_PATH)

with tf.device(DEVICE_NAME):
    y_pred_probs   = best_model.predict(X_test_pca, batch_size=BATCH_SIZE)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=CLASS_NAMES))

f1_weighted = f1_score(y_test, y_pred_classes, average='weighted')
f1_macro    = f1_score(y_test, y_pred_classes, average='macro')
print(f"Weighted F1-score : {f1_weighted:.4f}")
print(f"Macro    F1-score : {f1_macro:.4f}")

cm = confusion_matrix(y_test, y_pred_classes)
print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.matshow(cm, cmap=plt.cm.Blues)
plt.colorbar(im)
tick_labels = ['nv', 'mel', 'others']
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(tick_labels)
ax.set_yticklabels(tick_labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix (Test Set)\nWeighted F1 = {f1_weighted:.4f}')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

---
## 9. Final Evaluation Cell

*The cell below is prepared for the evaluation of the model on an unseen hidden dataset.*

*Set `HIDDEN_TEST_CSV` and `HIDDEN_TEST_IMG_DIR` to the correct paths before running.*

**Important:** The PCA transform fitted on the training set is loaded from `pca.pkl` and applied consistently here — the hidden test images are processed identically to the training images (RGB, 128×128, normalised, then PCA-transformed).

In [ ]:
# ==========================================
# HIDDEN TEST SET EVALUATION
# ==========================================
HIDDEN_TEST_CSV     = 'path/to/hidden_test_labels.csv'
HIDDEN_TEST_IMG_DIR = 'path/to/hidden_test_images'


def evaluate_hidden_dataset(
    csv_path,
    img_dir,
    model_path=MODEL_PATH,
    pca_path=PCA_PATH
):
    """
    Evaluate the saved model on a hidden test set.
    Applies the SAME preprocessing pipeline used during training:
      RGB → resize to IMG_WIDTH×IMG_HEIGHT → normalise → flatten → PCA.
    """
    try:
        final_model = load_model(model_path)
        with open(pca_path, 'rb') as f:
            pca_loaded = pickle.load(f)
    except FileNotFoundError as e:
        print(f"Could not load model or PCA: {e}")
        return

    try:
        hidden_df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"CSV not found: {csv_path}")
        return

    images, labels = [], []
    for _, row in hidden_df.iterrows():
        img_path = os.path.join(img_dir, row['image_id'] + '.jpg')
        if not os.path.exists(img_path):
            continue
        try:
            img = Image.open(img_path).convert('RGB')           
            img = img.resize((IMG_WIDTH, IMG_HEIGHT))
            arr = np.array(img, dtype=np.float32) / 255.0
            images.append(arr.flatten())
            labels.append(map_labels(row['dx']))
        except Exception as e:
            print(f"Error loading {img_path}: {e}")

    if not images:
        print("No images loaded — check HIDDEN_TEST_IMG_DIR and CSV image_id values.")
        return

    X_hidden_raw = np.array(images, dtype=np.float32)
    y_hidden     = np.array(labels, dtype=np.int32)

    # Apply the SAME PCA fitted on training data
    X_hidden_pca = pca_loaded.transform(X_hidden_raw)

    with tf.device(DEVICE_NAME):
        y_pred_probs   = final_model.predict(X_hidden_pca, batch_size=BATCH_SIZE)
    y_pred_classes = np.argmax(y_pred_probs, axis=1)

    print("\nClassification Report (Hidden Set):")
    print(classification_report(y_hidden, y_pred_classes, target_names=CLASS_NAMES))

    f1_w = f1_score(y_hidden, y_pred_classes, average='weighted')
    f1_m = f1_score(y_hidden, y_pred_classes, average='macro')
    print(f"Weighted F1-score : {f1_w:.4f}")
    print(f"Macro    F1-score : {f1_m:.4f}")

    cm_h = confusion_matrix(y_hidden, y_pred_classes)
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.matshow(cm_h, cmap=plt.cm.Blues)
    plt.colorbar(im)
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(['nv', 'mel', 'others'])
    ax.set_yticklabels(['nv', 'mel', 'others'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix (Hidden Set)\nWeighted F1 = {f1_w:.4f}')
    for i in range(cm_h.shape[0]):
        for j in range(cm_h.shape[1]):
            ax.text(j, i, str(cm_h[i, j]), ha='center', va='center',
                    color='white' if cm_h[i, j] > cm_h.max() / 2 else 'black')
    plt.tight_layout()
    plt.show()


evaluate_hidden_dataset(HIDDEN_TEST_CSV, HIDDEN_TEST_IMG_DIR)